# AI Bug Classifier - Model Training & Evaluation
This notebook demonstrates how to load the bug report dataset, preprocess the text, train a text classification model (using TF-IDF + Logistic Regression/Naive Bayes), and export the trained model artifacts (`classifier.pkl` and `vectorizer.pkl`) for use by the API microservice.

### 1. Import Dependencies
We use `pandas` for data manipulation, `scikit-learn` for feature extraction and model training, and `joblib` / `pickle` for model serialization.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

### 2. Load Datasets
Let's load the `train.csv` and `test.csv` dataset files.

In [ ]:
# Load training and testing data
train_df = pd.read_csv('../dataset/train.csv')
test_df = pd.read_csv('../dataset/test.csv')

print(f"Training set size: {len(train_df)} rows")
print(f"Testing set size: {len(test_df)} rows")
print("\nFirst 3 rows of training data:")
train_df.head(3)

### 3. Preprocess Text Data
We combine the bug `title` and `description` to create a feature representation text column.

In [ ]:
# Combine fields
train_df['text'] = train_df['title'].fillna('') + " " + train_df['description'].fillna('')
test_df['text'] = test_df['title'].fillna('') + " " + test_df['description'].fillna('')

X_train, y_train = train_df['text'], train_df['category']
X_test, y_test = test_df['text'], test_df['category']

### 4. TF-IDF Vectorization
Convert the textual data into numerical vectors using a TF-IDF Vectorizer.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', min_df=1, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Feature matrix shape: {X_train_vec.shape}")

### 5. Train Classifier
We train a Logistic Regression model as a baseline. Note that since our training sample size is small, we use standard hyperparameters.

In [ ]:
classifier = LogisticRegression(C=1.0, max_iter=200, solver='liblinear')
classifier.fit(X_train_vec, y_train)

### 6. Evaluate Model Performance
Compute accuracy and classification report metrics on the test dataset.

In [ ]:
y_pred = classifier.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

### 7. Export Model Artifacts
Save the vectorizer and classifier to the `model/` folder to overwrite the baseline heuristic models.

In [ ]:
model_dir = '../model'
os.makedirs(model_dir, exist_ok=True)

with open(os.path.join(model_dir, 'classifier.pkl'), 'wb') as f:
    pickle.dump(classifier, f)

with open(os.path.join(model_dir, 'vectorizer.pkl'), 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model training artifacts successfully exported to model/ folder!")